# Reception-to-Shot Conversion Analysis

This notebook analyzes whether teams that received the ball in dangerous defensive-line spaces during the 2026 FIFA World Cup actually converted those moments into shots.

The analysis uses team-level FIFA Match Centre data and focuses on the relationship between Reception Access (receptions between the midfield/defensive lines and behind the defensive line) and total shot attempts.

## Main Question

After a team received the ball in a dangerous area, how often did that moment actually turn into a shot?

## Key Ideas

- Reception Access measures how often a team received the ball in the most dangerous defensive-line spaces, not just how often it entered the final third.
- Reception-to-Shot Conversion measures how efficiently those dangerous receptions were turned into shot attempts.
- A team can receive the ball in dangerous areas often but still fail to convert that into shots.
- This builds on an earlier finding in this series: territory (Field Tilt) did not guarantee shot creation. This notebook narrows the question further, from "reaching the final third" down to "receiving the ball in the most dangerous pockets of space."

## Data Source

FIFA Match Centre, full FIFA Official Stats only.

Belgium vs Egypt is excluded from full-stat analysis because FIFA provides only Live Statistics for that match, not the full official stats section.

## Important Notes

- Rankings are descriptive, not causal.
- Match counts differ from 3 to 8, so smaller samples may be more volatile.
- xG and opponent strength are not controlled.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Optional label adjustment library
try:
    from adjustText import adjust_text
    ADJUST_TEXT_AVAILABLE = True
except ImportError:
    ADJUST_TEXT_AVAILABLE = False

# =========================================================
# Edit only this line if the project folder is moved
# =========================================================
BASE_DIR = Path.cwd().resolve()
for parent in [BASE_DIR, *BASE_DIR.parents]:
    if parent.name == "worldcup-2026-official-stats-analysis":
        BASE_DIR = parent
        break

DATA_DIR = BASE_DIR / "data" / "fifa_worldcup_2026" / "site_scrape"
RAW_CSV_PATH = DATA_DIR / "site_official_stats_team_wide_flagged.csv"

OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Input file:", RAW_CSV_PATH)
print("Output directory:", OUTPUT_DIR)
print("adjustText available:", ADJUST_TEXT_AVAILABLE)

In [ ]:
# =========================================================
# Step 2. Load the FIFA official stats dataset
# =========================================================

raw = pd.read_csv(RAW_CSV_PATH)

print("Raw dataset shape:", raw.shape)
print("Number of match-team rows:", len(raw))
print("Number of matches:", raw["match_id"].nunique())
print("Number of teams:", raw["team_name"].nunique())

display(raw.head())

In [ ]:
# =========================================================
# Step 2.5. Check for missing values before calculation
# =========================================================
# Before calculating anything, check whether the columns we need
# have missing values. Reception Access columns are tracking-based
# data, so we expect Belgium vs Egypt to be missing here (unlike
# the goal/shot columns used in the previous notebook, which were
# basic match records and had no missing values).

required_columns = [
    "stats_complete", "match_id", "team_side", "team_name",
    "opponent_side", "opponent_name", "stats_source",
    "attacking__attempts_at_goal__total",
    "attacking__offers_to_receive__receptions_between_midfield_and_defensive_lines",
    "attacking__offers_to_receive__receptions_behind_the_defensive_line",
]

total_rows = len(raw)
print(f"Total rows in dataset: {total_rows}")
print()

missing_counts = raw[required_columns].isna().sum()
print("Missing value count per column:")
print(missing_counts)

if missing_counts.sum() > 0:
    print("\n[INFO] Missing values found. Inspecting which rows are affected:")
    missing_rows = raw[raw[required_columns].isna().any(axis=1)]
    display(missing_rows[["team_name", "opponent_name", "stats_complete", "stats_source"]])
else:
    print(f"\n[OK] No missing values found among all {total_rows} rows — safe to proceed.")

In [ ]:
# =========================================================
# Step 3. Calculate Reception Access and shot conversion per team
# =========================================================
# What this step does:
# - Filter to matches where FIFA gave full official stats
#   (this drops Belgium vs Egypt, which is missing the Reception
#   Access columns we need).
# - For each team, add up Reception Access (receptions between the
#   midfield/defensive lines + receptions behind the defensive line)
#   and total shot attempts across all their matches.
# - reception_to_shot_ratio = shots ÷ Reception Access.
#   A HIGHER number means a team produced MORE shots for the same
#   number of dangerous receptions — i.e., it converted danger into
#   shots more efficiently.
#
# Why we sum first, then divide (instead of averaging per-match
# ratios): this gives more weight to matches with more events,
# which is a more reliable team-level number — same principle
# used throughout this series.

full_stats = raw[raw["stats_complete"] == True].copy()

team_metrics = full_stats.groupby("team_name", as_index=False).agg(
    matches_played=("match_id", "nunique"),
    total_shots=("attacking__attempts_at_goal__total", "sum"),
    recept_between=("attacking__offers_to_receive__receptions_between_midfield_and_defensive_lines", "sum"),
    recept_behind=("attacking__offers_to_receive__receptions_behind_the_defensive_line", "sum"),
)

team_metrics["reception_access"] = team_metrics["recept_between"] + team_metrics["recept_behind"]
team_metrics["reception_to_shot_ratio"] = team_metrics["total_shots"] / team_metrics["reception_access"]

print(f"Number of teams: {len(team_metrics)}")
print()

SEMIFINALISTS = ["Spain", "Argentina", "France", "England"]
sf_avg = team_metrics[team_metrics.team_name.isin(SEMIFINALISTS)]["reception_to_shot_ratio"].mean()
rest_avg = team_metrics[~team_metrics.team_name.isin(SEMIFINALISTS)]["reception_to_shot_ratio"].mean()
print(f"Semifinalists average: {sf_avg:.3f}")
print(f"Rest of field average: {rest_avg:.3f}")
print(f"Gap: {(sf_avg - rest_avg) / rest_avg * 100:.1f}%")

display(team_metrics.sort_values("reception_to_shot_ratio", ascending=False).head(10))

In [ ]:
# =========================================================
# Step 4. Scatter plot - Reception Access vs Shots per match
# =========================================================
# What this chart shows:
# - x-axis: how often a team received the ball in dangerous areas per match
# - y-axis: how many shots a team took per match
# - A team above the "average conversion line" turned dangerous
#   receptions into shots MORE efficiently than average.
#   (Correction from Step 3: a HIGHER reception_to_shot_ratio is
#   BETTER -- it means more shots came from the same number of
#   dangerous receptions, not fewer.)

SEMIFINALISTS = ["Spain", "Argentina", "France", "England"]

team_metrics["reception_access_per_match"] = team_metrics["reception_access"] / team_metrics["matches_played"]
team_metrics["shots_per_match"] = team_metrics["total_shots"] / team_metrics["matches_played"]

match_values = sorted(team_metrics["matches_played"].unique())
palette = ["#8b5cf6", "#3b82f6", "#10b981", "#f59e0b", "#dc2626"]
color_map = dict(zip(match_values, palette))
team_metrics["point_color"] = team_metrics["matches_played"].map(color_map)

min_m, max_m = team_metrics["matches_played"].min(), team_metrics["matches_played"].max()
team_metrics["point_size"] = 38 + (team_metrics["matches_played"] - min_m) / (max_m - min_m) * (145 - 38)

fig, ax = plt.subplots(figsize=(13, 9), dpi=150)
ax.scatter(
    team_metrics["reception_access_per_match"], team_metrics["shots_per_match"],
    s=team_metrics["point_size"], c=team_metrics["point_color"],
    edgecolors="white", linewidth=0.7, alpha=0.9, zorder=3,
)

texts = []
for _, row in team_metrics.iterrows():
    is_sf = row["team_name"] in SEMIFINALISTS
    texts.append(
        ax.text(
            row["reception_access_per_match"], row["shots_per_match"], row["team_name"],
            fontsize=8.5 if is_sf else 7, fontweight="bold" if is_sf else "normal",
            color="#111827", zorder=5 if is_sf else 4,
        )
    )

adjust_text(
    texts, ax=ax, expand=(1.6, 1.9), force_text=(0.9, 1.2), force_static=(0.5, 0.7),
    force_pull=(0.01, 0.01), max_move=(60, 60), min_arrow_len=3,
    arrowprops=dict(arrowstyle="-", color="#475569", lw=0.6, alpha=0.7), iter_lim=3000,
)

ax.set_xlabel("Reception Access per Match (dangerous receptions)", fontsize=11)
ax.set_ylabel("Shots per Match", fontsize=11)
ax.set_title("Did Dangerous Receptions Turn Into Shots?", fontsize=15, pad=14)
ax.grid(alpha=0.2)
for spine in ax.spines.values():
    spine.set_visible(False)

from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map[mc], markeredgecolor="white",
           markersize=10, label=f"{mc} matches" + (" (Semifinalists)" if mc == max_m else ""))
    for mc in match_values
]
ax.legend(handles=legend_handles, loc="upper left", fontsize=8.5, frameon=True,
          facecolor="white", edgecolor="#d1d5db", title="Matches played")

fig.text(
    0.08, 0.02,
    "Semifinalists converted dangerous receptions into shots slightly more efficiently on average "
    "(shots per reception: 0.127 vs 0.116 for the rest of the field).\n"
    "Belgium vs Egypt excluded (missing tracking data for this match). Source: FIFA Match Centre | Full Official Stats only.",
    fontsize=8, color="gray",
)

plt.tight_layout(rect=[0, 0.06, 1, 1])

output_path = OUTPUT_DIR / "07_reception_to_shot_conversion.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight")
plt.show()
print(f"Saved: {output_path}")


In [ ]:
# =========================================================
# Step 5. Calculate Box Shot Share and Box Conversion Rate
# =========================================================
# What this step adds:
# - Box Shot Share: of all the shots a team took, what share came
#   from inside the penalty area (the highest-quality shooting zone).
# - Box Conversion Rate: of the shots taken inside the box, what
#   share actually became goals. This is the closest thing to a
#   "finishing quality" measure we can build without xG data.
#
# Caution: Box Conversion Rate is goal-based, and goals are rare
# events. A team with very few total shots can have this percentage
# swing sharply from one extra goal. To check whether the pattern
# is real and not just small-sample noise, we also recompute it
# using only teams with 4+ matches (a larger, more stable sample).

team_metrics["box_shot_share"] = full_stats.groupby("team_name")["attacking__attempts_at_goal__inside_the_penalty_area"].sum().values / team_metrics["total_shots"] * 100

box_goals_by_team = full_stats.groupby("team_name")["attacking__goal__inside_the_penalty_area"].sum()
box_attempts_by_team = full_stats.groupby("team_name")["attacking__attempts_at_goal__inside_the_penalty_area"].sum()

team_metrics["box_goals"] = team_metrics["team_name"].map(box_goals_by_team)
team_metrics["box_attempts"] = team_metrics["team_name"].map(box_attempts_by_team)
team_metrics["box_conversion_rate"] = team_metrics["box_goals"] / team_metrics["box_attempts"] * 100

SEMIFINALISTS = ["Spain", "Argentina", "France", "England"]

sf_avg = team_metrics[team_metrics.team_name.isin(SEMIFINALISTS)]["box_conversion_rate"].mean()
rest_avg = team_metrics[~team_metrics.team_name.isin(SEMIFINALISTS)]["box_conversion_rate"].mean()
print(f"All 48 teams — Semifinalists: {sf_avg:.1f}%, Rest: {rest_avg:.1f}%, Gap: {(sf_avg - rest_avg) / rest_avg * 100:.1f}%")

# Robustness check: same comparison, teams with 4+ matches only
robust = team_metrics[team_metrics.matches_played >= 4]
sf_robust = robust[robust.team_name.isin(SEMIFINALISTS)]["box_conversion_rate"].mean()
rest_robust = robust[~robust.team_name.isin(SEMIFINALISTS)]["box_conversion_rate"].mean()
print(f"4+ matches only ({len(robust)} teams) — Semifinalists: {sf_robust:.1f}%, Rest: {rest_robust:.1f}%, Gap: {(sf_robust - rest_robust) / rest_robust * 100:.1f}%")

display(team_metrics.sort_values("box_conversion_rate", ascending=False)[["team_name", "matches_played", "box_attempts", "box_goals", "box_conversion_rate"]].head(10))

In [ ]:
# =========================================================
# Step 6. Scatter plot - Box Conversion Rate vs sample size
# =========================================================
# What this chart shows:
# - x-axis: how many shots a team took from inside the box
#   (this doubles as a sample-size signal)
# - y-axis: Box Conversion Rate (%)
# - Point size and color also reflect matches played, for a
#   second visual cue about sample size.
#
# Why this layout: teams on the LEFT (few box attempts) can swing
# sharply from a single goal, so their extreme percentages (very
# high or very low) should be read with caution. Semifinalists
# (large red dots) sit on the RIGHT with a large sample, and their
# conversion rate holds up around or above the tournament average
# -- unlike the top-ranked teams by raw percentage (Tunisia, Japan),
# whose small sample makes their numbers less reliable.

SEMIFINALISTS = ["Spain", "Argentina", "France", "England"]

match_values = sorted(team_metrics["matches_played"].unique())
palette = ["#8b5cf6", "#3b82f6", "#10b981", "#f59e0b", "#dc2626"]
color_map = dict(zip(match_values, palette))
team_metrics["point_color"] = team_metrics["matches_played"].map(color_map)

min_m, max_m = team_metrics["matches_played"].min(), team_metrics["matches_played"].max()
team_metrics["point_size"] = 38 + (team_metrics["matches_played"] - min_m) / (max_m - min_m) * (145 - 38)

fig, ax = plt.subplots(figsize=(13, 9), dpi=150)
ax.scatter(
    team_metrics["box_attempts"], team_metrics["box_conversion_rate"],
    s=team_metrics["point_size"], c=team_metrics["point_color"],
    edgecolors="white", linewidth=0.7, alpha=0.9, zorder=3,
)

texts = []
for _, row in team_metrics.iterrows():
    is_sf = row["team_name"] in SEMIFINALISTS
    texts.append(
        ax.text(
            row["box_attempts"], row["box_conversion_rate"], row["team_name"],
            fontsize=8.5 if is_sf else 7, fontweight="bold" if is_sf else "normal",
            color="#111827", zorder=5 if is_sf else 4,
        )
    )

adjust_text(
    texts, ax=ax, expand=(1.6, 1.9), force_text=(0.9, 1.2), force_static=(0.5, 0.7),
    force_pull=(0.01, 0.01), max_move=(60, 60), min_arrow_len=3,
    arrowprops=dict(arrowstyle="-", color="#475569", lw=0.6, alpha=0.7), iter_lim=3000,
)

ax.axhline(team_metrics["box_conversion_rate"].mean(), linestyle="--", color="gray", linewidth=1, alpha=0.7)
ax.text(team_metrics["box_attempts"].max() * 0.98, team_metrics["box_conversion_rate"].mean() + 0.5,
        "Tournament average", fontsize=8, color="gray", ha="right")

ax.set_xlabel("Box Attempts (total shots taken inside the penalty area)", fontsize=11)
ax.set_ylabel("Box Conversion Rate (%)", fontsize=11)
ax.set_title("Finishing Quality Inside the Box — and How Much to Trust It", fontsize=15, pad=14)
ax.grid(alpha=0.2)
for spine in ax.spines.values():
    spine.set_visible(False)

from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map[mc], markeredgecolor="white",
           markersize=10, label=f"{mc} matches" + (" (Semifinalists)" if mc == max_m else ""))
    for mc in match_values
]
ax.legend(handles=legend_handles, loc="upper right", fontsize=8.5, frameon=True,
          facecolor="white", edgecolor="#d1d5db", title="Matches played")

fig.text(
    0.08, 0.02,
    "Teams on the left (few box attempts) can swing sharply from a single goal -- treat their high or low\n"
    "percentages with caution. Semifinalists' box conversion rate held up even among the larger-sample teams "
    "(13.0% gap vs rest, for teams with 4+ matches).\n"
    "Belgium vs Egypt excluded (missing tracking data for this match). Source: FIFA Match Centre | Full Official Stats only.",
    fontsize=8, color="gray",
)

plt.tight_layout(rect=[0, 0.08, 1, 1])

output_path = OUTPUT_DIR / "07_box_conversion_rate.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight")
plt.show()
print(f"Saved: {output_path}")

In [ ]:
ㅌ